In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt

0. Seeting the random seed

In [2]:
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)         # Cover the GPU

1. Using nerual network to parameteraztion (CNN as a case)

In [3]:
# 1D CNN, in order to generate the five parameter curves (varying with depth)
# Input: Z_normal (1D vector, length of N); Output: [N,5] vector

class ParamNet1D(nn.Module):
    def __init__(self):
        super().__init__()
        # two CNN layers (input 1 channel → hidden → out_channels )
        self.conv1 = nn.Conv1d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(32, 32,  kernel_size=3, padding=1)
        self.conv3 = nn.Conv1d(32, 4,  kernel_size=3, padding=1)
    
    def forward(self, z_norm):
        # add the dimension of input
        x = z_norm.unsqueeze(0).unsqueeze(0)  # (1,1,N)  (B=1, C=1, N)
        x = F.relu(self.conv1(x))             # (1,32,N) (1, hidden, N)
        x = F.relu(self.conv2(x))             # (32,32,N) (1, hidden, N)
        x = torch.tanh(self.conv3(x))         # (1,4,N)  (1, 4, N)
        # reduce the dimension of batch, (N,4)
        amp = x.squeeze(0).permute(1,0)       # (N,4)
        return amp

# Note: One can select other NN types to conduct the parameteraztion of ground thermodynamic properties

2. DM inverse model

In [4]:
class InversionModel(nn.Module):
    def __init__(self, z, dz, dt_day, prior_means, prior_stds):
        """
        z: Tensor[N] depth
        dz: float spatial step
        dt_day: float temperal step
        """
        super().__init__()
        self.z      = z                                  # Depth vector
        self.N      = z.numel()
        self.dz     = dz
        self.dt     = dt_day                             # [day]
        self.freq   = pd.to_timedelta(dt_day, unit='D')  # Timedelta (dt_day = 1 day)

        # prior
        self.register_buffer('prior_means', prior_means.view(1,4)) # (N,4)
        self.register_buffer('prior_stds',  prior_stds.view(1,4))

        self.param_net = ParamNet1D()   # NN interface

        # Two uncertain n-factors (need to be trained)
        self.n_sum = nn.Parameter(torch.tensor(0.5))
        self.n_win = nn.Parameter(torch.tensor(0.5))
        self.a_raw = nn.Parameter(torch.tensor(-0.2231)) 

        # If directly use the soil surface temp as the upper boundary condition
        self.use_direct_surface = False
    
    def a_effective(self):
        return 0.01 + 0.09 * torch.sigmoid(self.a_raw)
    

    def forward(self, T_weather, bc_dates, T_init):
        """
        T_weather: numpy array [nt_days] 
        bc_dates:  DatetimeIndex len: nt_days
        """
        device = self.z.device
        dtype = self.z.dtype

        # 1) Generate constitutive parameters
        z_min, z_max = self.z.min(), self.z.max()
        z_norm = (self.z - z_min)/(z_max - z_min)
        # z_in = z_norm.unsqueeze(0).unsqueeze(0); Tensor shape (1,1,N)
        amp = self.param_net(z_norm)          # (N,5)
        params = self.prior_means + amp * self.prior_stds  # (N,4)
        theta_sat, tao, Ks, Cvs = params.unbind(dim=1)

        a_raw_safe = torch.nan_to_num(self.a_raw, nan=0.0, posinf=5.0, neginf=-5.0) 
        a = 0.01 + 0.09 * torch.sigmoid(a_raw_safe)
        theta_res = a * theta_sat


        # 2) Generate T_surface
        if isinstance(T_weather, torch.Tensor):
           T_up = T_weather.to(device, dtype=dtype)
        else:
           T_up = torch.from_numpy(T_weather).to(device, dtype=dtype)

        if self.use_direct_surface:                # Directly use the soil temp as the upper boundary condition
            T_up = T_up
        else:
            n_sum = self.n_sum.clamp(0,1)
            n_win = self.n_win.clamp(0,1)
            T_surf_daily =  torch.where(T_up>=0, T_up*n_sum, T_up*n_win)
            T_up = T_surf_daily


        # 3) generate dates
        nt = T_up.shape[0]
        dates = pd.date_range(start=bc_dates[0], periods=nt, freq=self.freq)


        # 4) Heat transfer model for soil
        T = T_init.clone().to(device)                                  # Initial condition (N,) 1D tensor
        T_hist = torch.zeros(nt, self.N, device=device, dtype=dtype)   # (Nt, Nz) 2D tensor

        #dz = self.dz
        dt = self.dt


        # Define the constitutive relationships
        class TmpModel: pass
        M = TmpModel()
        M.z         = self.z
        M.theta_sat = theta_sat
        M.theta_res = theta_res
        M.tao       = tao
        M.Ks        = Ks
        M.Cvs       = Cvs
        M.Cvw, M.Cvi  = 4.19e6, 2.13e6
        M.Li        = 3.34e8
        M.Kw, M.Ki  = 0.56*(24*3600), 2.26*(24*3600)   # J/(m·K·day)

        def theta_w(T):
            return torch.where(
                T>=0,
                M.theta_sat,
                M.theta_res + (M.theta_sat-M.theta_res)*torch.exp(T/M.tao)
            )

        def dtheta_w_dT(T):
            return torch.where(
                T>=0,
                torch.zeros_like(T),
                (M.theta_sat-M.theta_res)*torch.exp(T/M.tao)/M.tao
            )

        def K_eq(T):
            tw   = theta_w(T)
            phi_s = 1 - M.theta_sat
            return M.Ks**phi_s * M.Kw**tw * M.Ki**(M.theta_sat - tw)

        def Cv_eq(T):
            tw = theta_w(T)
            dθdT = dtheta_w_dT(T)
            phi_s = 1 - M.theta_sat
            base = M.Cvs*phi_s + M.Cvw*tw + M.Cvi*(M.theta_sat - tw)
            return base + M.Li*dθdT
        

        # FDM solver (implicit version)
        def _solve_tridiag_dense(a, b, c, d):
            Ni = b.numel()
            LHS = torch.zeros(Ni, Ni, dtype=dtype, device=device)
            idx = torch.arange(Ni, device=device)
            LHS[idx, idx] = b
            if Ni > 1:
                LHS[idx[1:], idx[:-1]] = a[1:]   
                LHS[idx[:-1], idx[1:]] = c[:-1] 
            x = torch.linalg.solve(LHS, d.unsqueeze(1)).squeeze(1)
            return x

        def implicit_theta_step(Tn, T_top_n, T_top_np1, theta=0.5, picard_iters=2):
            N  = Tn.numel()
            Ni = N - 1

            dz_face = M.z[1:] - M.z[:-1]               # [N-1]  Δz_{i+1/2}
            dz_up   = dz_face[:Ni]                     # [Ni]   Δz_{i-1/2}
            dz_dn   = torch.empty_like(dz_up)          # [Ni]   Δz_{i+1/2}
            if Ni > 1:
                dz_dn[:-1] = dz_face[1:Ni]
            dz_dn[-1] = 1.0                            

            dzcv = torch.empty(Ni, dtype=dtype, device=device)  
            if Ni > 1:
                dzcv[:-1] = 0.5 * (M.z[2:] - M.z[:-2])
            dzcv[-1] = 0.5 * (M.z[-1] - M.z[-2])       

            Tguess = Tn.clone()

            for _ in range(picard_iters):
                Tn_full         = Tn.clone()           # [N]
                Tn_full[0]      = T_top_n              
                Tnp1_full       = Tguess.clone()       # [N]
                Tnp1_full[0]    = T_top_np1            

                Cv_full = Cv_eq(Tnp1_full)             # [N]
                K_node_full = K_eq(Tnp1_full)          # [N]

                K_face = 2.0 * K_node_full[1:] * K_node_full[:-1] / (K_node_full[1:] + K_node_full[:-1] + 1e-12)
                Kup = K_face[:Ni]                      # [Ni] i-1/2
                Kdn = torch.empty_like(Kup)            # [Ni] i+1/2
                if Ni > 1:
                    Kdn[:-1] = K_face[1:Ni]
                Kdn[-1] = 0.0                         

                Cv_i   = Cv_full[1:]                   # [Ni]
                coef   = dt / (Cv_i * dzcv)            # [Ni]
                lam_up = coef * (Kup / dz_up)          # [Ni]
                lam_dn = coef * (Kdn / dz_dn)          # [Ni]

                a = -theta * lam_up
                b =  1.0 + theta * (lam_up + lam_dn)
                c = -theta * lam_dn

                Tn_i = Tn_full[1:]                     # [Ni]
                up_n = Tn_full[:-1]                    # [Ni]
                dn_n = torch.empty_like(Tn_i)          # [Ni]
                if Ni > 1:
                    dn_n[:-1] = Tn_full[2:]
                dn_n[-1]  = Tn_full[-1]

                rhs = Tn_i + (1.0 - theta) * (
                    lam_up * (up_n - Tn_i) - lam_dn * (Tn_i - dn_n)
                )
            
                rhs[0] += theta * lam_up[0] * T_top_np1

                
                X = _solve_tridiag_dense(a, b, c, rhs)
                Tnew = Tn.clone()
                Tnew[0]  = T_top_np1
                Tnew[1:] = X

                if torch.max(torch.abs(Tnew - Tguess)) < 1e-5:
                    Tguess = Tnew
                    break
                Tguess = Tnew

            return Tguess

        T_hist[0].copy_(T)
        for d in range(nt - 1):
           T_top_n   = T_up[d].to(T)
           T_top_np1 = T_up[d+1].to(T)
           T = implicit_theta_step(T, T_top_n, T_top_np1, theta=0.5, picard_iters=2)
           T_hist[d+1].copy_(T)

        return T_hist, dates
    

3. Input data

In [5]:
# (1) Upper boundary condition
gt_df_all = pd.read_excel(r"D:/UW Madsion/Permafrost thermodynamic state prediction/Delta HT4S model/Applications/Borrow site/In situ data/T_soil data(Barrow site).xlsx", index_col=0, parse_dates=True).sort_index()
depth_vals_all = np.array([float(str(c).replace("m","")) for c in gt_df_all.columns])
zero_col_idx = int(np.argmin(np.abs(depth_vals_all - 0.0)))
col0 = gt_df_all.columns[zero_col_idx]

# Separate training and test dataset
start, end = pd.to_datetime("2006-10-18"), pd.to_datetime("2007-12-31")
train_mask = (gt_df_all.index >= start) & (gt_df_all.index <= end)
days_train = gt_df_all.index[train_mask]
days_test  = gt_df_all.index[~train_mask]
T_up_train = gt_df_all[col0].reindex(days_train).values
T_up_test  = gt_df_all[col0].reindex(days_test ).values


# (2) Initial condition (T_int)
ic_df = pd.read_excel(r"D:/UW Madsion/Permafrost thermodynamic state prediction/Delta HT4S model/Applications/Borrow site/In situ data/Initial condition(Barrow).xlsx", usecols=[0,1], names=['z','T'], header=0)
z_full      = torch.tensor(ic_df['z'].values, dtype=torch.float32)  # [Nz]
T_init_full = torch.tensor(ic_df['T'].values, dtype=torch.float32)  # [Nz]
Nz = z_full.numel()


# (3) Ground truth
cols_rest = [c for i,c in enumerate(gt_df_all.columns) if i != zero_col_idx]
gt_df = gt_df_all[cols_rest] 
# "0.25 m" to "0.25"
depth_vals_rest = np.array([float(str(c).replace("m","")) for c in cols_rest])
sparse_idxs = [int(round(d / 0.25)) for d in depth_vals_rest]  
# Training/test separate
#start, end = pd.to_datetime("2006-12-01"), pd.to_datetime("2009-04-01")
#mask_full = (gt_sparse.index.normalize() >= start) & (gt_sparse.index.normalize() <= end)
gt_train = gt_df.reindex(days_train)
gt_test  = gt_df.reindex(days_test)


# (4) Transfer to tensors
device = 'cuda' if torch.cuda.is_available() else 'cpu'
T_up_train_t = torch.from_numpy(T_up_train).float().to(device)
T_up_test_t  = torch.from_numpy(T_up_test).float().to(device)
T_init_full     = T_init_full.to(device)
z_full          = z_full.to(device)
gt_train_t = torch.from_numpy(gt_train.values).float().to(device)
gt_test_t  = torch.from_numpy(gt_test.values).float().to(device)


print("Sparse depth:", gt_df.columns.tolist())
print("Training dataset shape:", gt_train.shape, "Testing dataset shape:", gt_test.shape)
print(sparse_idxs)

print("gt_train_t NaN?", torch.isnan(gt_train_t).any().item())
print("gt_test_t NaN?", torch.isnan(gt_test_t).any().item())
print("weather_train_t NaN?", torch.isnan(T_up_train_t).any().item())
print("T_init_full NaN?", torch.isnan(T_init_full).any().item())

print(T_up_train)


Sparse depth: ['1', '2', '3', '4', '5', '6']
Training dataset shape: (440, 6) Testing dataset shape: (360, 6)
[4, 8, 12, 16, 20, 24]
gt_train_t NaN? False
gt_test_t NaN? True
weather_train_t NaN? False
T_init_full NaN? False
[-1.6700e+00 -2.0720e+00 -1.7840e+00 -1.8420e+00 -1.5840e+00 -2.0720e+00
 -2.3330e+00 -2.0720e+00 -2.0440e+00 -2.1880e+00 -2.4490e+00 -2.5370e+00
 -2.1880e+00 -3.4190e+00 -5.2940e+00 -2.8290e+00 -1.9860e+00 -2.8290e+00
 -1.8710e+00 -2.0720e+00 -2.6530e+00 -3.1530e+00 -3.2120e+00 -4.6510e+00
 -3.8660e+00 -5.3250e+00 -4.9250e+00 -7.5100e+00 -7.4770e+00 -7.9340e+00
 -9.8500e+00 -8.9650e+00 -1.3580e+01 -1.1830e+01 -9.8840e+00 -1.0756e+01
 -1.1794e+01 -1.0545e+01 -1.3201e+01 -1.3849e+01 -1.2195e+01 -1.3087e+01
 -1.2601e+01 -1.2232e+01 -9.4390e+00 -6.1360e+00 -7.8360e+00 -8.7630e+00
 -7.2830e+00 -6.8010e+00 -8.1320e+00 -1.0792e+01 -1.1433e+01 -1.0335e+01
 -1.0196e+01 -1.0827e+01 -1.0686e+01 -1.0615e+01 -1.0127e+01 -1.0686e+01
 -1.1721e+01 -1.2012e+01 -1.1685e+01 -1.2012e

4. MAP Training loop

In [ ]:
# (1) Model parameter setting
factor = 1
dt_day = 1.0

prior_means = torch.tensor([0.5, 3.0, 7.0*24*3600, 4.0e6], dtype=torch.float32).to(T_up_train_t.device)
prior_stds  = torch.tensor([0.3, 2.5, 6.8*24*3600, 3.8e6], dtype=torch.float32).to(T_up_train_t.device)

inv_model = InversionModel(
    z_full, dz=0.25, dt_day = dt_day,
    prior_means = prior_means,
    prior_stds = prior_stds).to(device)


inv_model.use_direct_surface = True

for p in [inv_model.n_sum, inv_model.n_win]:
    p.requires_grad_(False)

opt = torch.optim.Adam([p for p in inv_model.parameters() if p.requires_grad], lr=1e-3)
#loss_fn = nn.MSELoss()

def mse_ignore_nan(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    tmask = torch.isfinite(target)
    if not torch.any(tmask):
        return torch.zeros((), device=pred.device, dtype=pred.dtype)
    if not torch.isfinite(pred[tmask]).all():
        return torch.tensor(1e6, device=pred.device, dtype=pred.dtype)  
    diff = pred[tmask] - target[tmask]
    return (diff * diff).mean()


# -------------       Check if all parameters are optimized by Adam       -----------------
need_train = [(n,p) for n,p in inv_model.named_parameters() if p.requires_grad]
opt_ids = {id(p) for g in opt.param_groups for p in g['params']}
missing = [n for n,p in need_train if id(p) not in opt_ids]
print("Missing-from-optimizer:", missing)
# -----------------------------------------------------------------------------------------


# (2) Map training loop
from copy import deepcopy

train_losses, test_losses = [], []

for epoch in range(1001):
    inv_model.train()
    opt.zero_grad()
    # forward modeling
    sim_train, _ = inv_model(T_up_train_t, days_train, T_init_full)
    # sparse sampling
    sim_sparse_train = sim_train[:, sparse_idxs]
    loss_train   = mse_ignore_nan(sim_sparse_train, gt_train_t)

    if not torch.isfinite(loss_train):
        print("[WARN] non-finite train loss; skip this step.")
        opt.zero_grad(set_to_none=True)
        with torch.no_grad():
            inv_model.a_raw.data = torch.nan_to_num(inv_model.a_raw.data, nan=0.0, posinf=5.0, neginf=-5.0)
        continue

    loss_train.backward()
    opt.step()

    with torch.no_grad():
        inv_model.a_raw.data = torch.nan_to_num(inv_model.a_raw.data, nan=0.0, posinf=5.0, neginf=-5.0)

    # Testing
    inv_model.eval()
    with torch.no_grad():
        T0_test = sim_train[-1].detach()           # shape: (Nz,) 1D vector
        sim_test, _ = inv_model(T_up_test_t, days_test, T0_test)
        sim_sparse_test = sim_test[:, sparse_idxs]
        loss_test   = mse_ignore_nan(sim_sparse_test, gt_test_t)
    
    tr, te = loss_train.item(), loss_test.item()
    a_eff = float(inv_model.a_effective().detach().cpu()) 
    train_losses.append(tr)
    test_losses.append(te)
    print(f"Epoch {epoch:04d} | Train MSE: {tr:.6f} | Test MSE: {te:.6f} | a_eff: {a_eff:.5f}")


print(f"Finished training. Using final checkpoint at epoch {epoch}.")
    


# (3) Loss function curve
import os, numpy as np, pandas as pd
# save MAP loss data
os.makedirs("exports_MAP loss", exist_ok=True)
epochs = np.arange(len(train_losses))
df_loss = pd.DataFrame({"epoch": epochs,"train_mse": train_losses,"test_mse":  test_losses})
df_loss.to_csv("exports_MAP loss/losses.csv", index=False)
print("Save to: exports_MAP loss/losses.csv")

import matplotlib.pyplot as plt
plt.figure(figsize=(6,4))
plt.plot(range(len(train_losses)), train_losses, label="Train MSE")
plt.plot(range(len(test_losses)),  test_losses,  label="Test MSE")
plt.xlabel("Epoch")
plt.ylabel("Loss (MSE)")
plt.title("Training vs Test Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Missing-from-optimizer: []


5. MAP data save

In [ ]:
os.makedirs("exports_MAP visualization", exist_ok=True)
OUT_DIR = "exports_MAP visualization" 

def expand_subdaily_index(days_idx, factor):
    days_idx = pd.to_datetime(days_idx)
    step_min = 1440 / factor                    
    offsets = pd.to_timedelta(np.arange(factor) * step_min, unit='m')
    arr = (days_idx.values[:, None] + offsets.values[None, :]).ravel()
    return pd.DatetimeIndex(arr)

@torch.no_grad()
def _get_cnn_param_curves(model, z_full):
    z_norm = (z_full - z_full.min()) / (z_full.max() - z_full.min() + 1e-8)  # [Nz]
    out = model.param_net(z_norm)
    amp = out[0] if isinstance(out, (tuple, list)) else out
    if amp.ndim == 3:            
        amp = amp.squeeze(0).T   # -> [Nz, 4]
    elif amp.ndim == 2 and amp.shape[0] == 5:   # [4, Nz]
        amp = amp.T                               # -> [Nz, 4]
    return amp * model.prior_stds + model.prior_means   # [Nz, 4]

@torch.no_grad()
def _n_eff(m):
    return float(m.n_sum.clamp(0,1)), float(m.n_win.clamp(0,1))


curves = _get_cnn_param_curves(inv_model, z_full).cpu().numpy()  # [N,4]
depth  = z_full.detach().cpu().numpy().ravel()
pd.DataFrame(curves, index=depth, columns=["theta_sat","tau","Ks","Cv"]).to_csv(f"{OUT_DIR}/cnn_curves.csv", index_label="depth_m")

with torch.no_grad():
    a_eff = float(inv_model.a_effective().detach().cpu())
print(f"[INFO] a (effective for theta_res = a * theta_sat) = {a_eff:.6f}")
pd.DataFrame({"a_eff": [a_eff]}).to_csv(f"{OUT_DIR}/a_value.csv", index=False)


n_sum_eff, n_win_eff = _n_eff(inv_model)
pd.DataFrame({"n_sum":[n_sum_eff], "n_win":[n_win_eff]}).to_csv(f"{OUT_DIR}/n_factors.csv", index=False)              
print(f"n_sum(eff)={n_sum_eff:.4f}, n_win(eff)={n_win_eff:.4f}")


inv_model.eval()
with torch.no_grad():
    sim_train, _ = inv_model(T_up_train_t, days_train, T_init_full)
    T0_test = sim_train[-1]
    sim_test,  _ = inv_model(T_up_test_t,  days_test,  T0_test)

time_train = expand_subdaily_index(days_train, factor)
time_test  = expand_subdaily_index(days_test,  factor)

cols_full = [f"{d:.2f}m" for d in depth]

sim_train_np = sim_train.cpu().numpy()
sim_test_np  = sim_test.cpu().numpy()
pd.DataFrame(sim_train_np, index=time_train, columns=cols_full).to_csv(f"{OUT_DIR}/train_sim_full.csv", index_label="time")
pd.DataFrame(sim_test_np,  index=time_test,  columns=cols_full).to_csv(f"{OUT_DIR}/test_sim_full.csv",  index_label="time")
np.save(f"{OUT_DIR}/train_sim_full.npy", sim_train_np)
np.save(f"{OUT_DIR}/test_sim_full.npy",  sim_test_np)


z_sparse = z_full[sparse_idxs].detach().cpu().numpy()
cols_sparse = [f"{d:.2f}m" for d in z_sparse]
pd.DataFrame(gt_train_t.detach().cpu().numpy(), index=time_train, columns=cols_sparse).to_csv(f"{OUT_DIR}/train_gt_sparse.csv", index_label="time")
pd.DataFrame(gt_test_t.detach().cpu().numpy(), index=time_test, columns=cols_sparse).to_csv(f"{OUT_DIR}/test_gt_sparse.csv", index_label="time")


pd.DataFrame({"T_surface": T_up_train_t.cpu().numpy()}, index=days_train).to_csv(f"{OUT_DIR}/surface_train.csv", index_label="day")
pd.DataFrame({"T_surface": T_up_test_t.cpu().numpy()}, index=days_test).to_csv(f"{OUT_DIR}/surface_test.csv", index_label="day")   

# 网格
np.save(f"{OUT_DIR}/z_full.npy", depth)                          
np.save(f"{OUT_DIR}/z_sparse.npy", z_sparse)        

[INFO] a (effective for theta_res = a * theta_sat) = 0.038542
n_sum(eff)=0.5000, n_win(eff)=0.5000
